<a href="https://colab.research.google.com/github/Adilonapsh/MicrosoftRoadDetectionsTSVGeojsonParser/blob/main/docs/examples/train_object_detection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train an Object Detection Model with GeoAI

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/train_object_detection_model.ipynb)

## Install package
To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [1]:
%pip install geoai-py

## Import libraries

In [2]:
import geoai

## Download sample data

In [3]:
train_raster_url = (
    "/content/Export_BUILDING.tif"
    # "/content/tegar-foto_udara_16052024.tif"
)
train_vector_url = "/content/train_mulyaharja_geojson.geojson"
test_raster_url = (
    "/content/Export_BUILDING.tif"
)

In [4]:
train_raster_path = geoai.download_file(train_raster_url)
train_vector_path = geoai.download_file(train_vector_url)
test_raster_path = geoai.download_file(test_raster_url)

File already exists: Export_BUILDING.tif
File already exists: train_mulyaharja_geojson.geojson
File already exists: Export_BUILDING.tif


## Visualize sample data

In [5]:
geoai.view_vector_interactive(train_vector_path, tiles=train_raster_url)

In [9]:
geoai.view_raster(test_raster_url)

In [8]:
out_folder = "output"
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_vector_path,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)


Raster info for Export_BUILDING.tif:
  CRS: EPSG:4326
  Dimensions: 10836 x 6402
  Resolution: (3.448056478412863e-07, 3.4724523586376605e-07)
  Bands: 4
  Bounds: BoundingBox(left=106.784940884, bottom=-6.650445798, right=106.788677198, top=-6.648222734)
Loaded 161 features from train_mulyaharja_geojson.geojson
Vector CRS: EPSG:4326



Generated: 1050, With features: 402: 100%|██████████| 1050/1050 [00:22<00:00, 47.23it/s]



------- Export Summary -------
Total tiles exported: 1050
Tiles with features: 402 (38.3%)
Average feature pixels per tile: 88879.5
Output saved to: output

------- Georeference Verification -------


## Train object detection model

In [11]:
geoai.train_MaskRCNN_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/models",
    num_channels=4,
    pretrained=True,
    batch_size=3,
    num_epochs=10,
    learning_rate=0.005,
    val_split=0.2,
)

Using device: cuda
Found 1050 image files and 1050 label files
Training on 840 images, validating on 210 images
Epoch: 0, Batch: 0/280, Loss: 3.2589, Time: 3.93s
Epoch: 0, Batch: 10/280, Loss: 1.3331, Time: 6.65s
Epoch: 0, Batch: 20/280, Loss: 1.1431, Time: 6.44s
Epoch: 0, Batch: 30/280, Loss: 0.0224, Time: 6.41s
Epoch: 0, Batch: 40/280, Loss: 0.7710, Time: 6.39s
Epoch: 0, Batch: 50/280, Loss: 0.7801, Time: 6.45s
Epoch: 0, Batch: 60/280, Loss: 0.9579, Time: 6.86s
Epoch: 0, Batch: 70/280, Loss: 1.1439, Time: 6.52s
Epoch: 0, Batch: 80/280, Loss: 0.0319, Time: 6.59s
Epoch: 0, Batch: 90/280, Loss: 0.0100, Time: 6.49s
Epoch: 0, Batch: 100/280, Loss: 0.8194, Time: 6.64s
Epoch: 0, Batch: 110/280, Loss: 0.7954, Time: 6.79s
Epoch: 0, Batch: 120/280, Loss: 0.0216, Time: 6.73s
Epoch: 0, Batch: 130/280, Loss: 0.7937, Time: 6.72s
Epoch: 0, Batch: 140/280, Loss: 0.9147, Time: 6.88s
Epoch: 0, Batch: 150/280, Loss: 1.0322, Time: 6.87s
Epoch: 0, Batch: 160/280, Loss: 0.8413, Time: 6.91s
Epoch: 0, Batch

## Run inference

In [12]:
masks_path = "naip_test_prediction.tif"
model_path = f"{out_folder}/models/best_model.pth"

In [13]:
geoai.object_detection(
    test_raster_path,
    masks_path,
    model_path,
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
    num_channels=4,
)

Processing 1050 windows with size 512x512 and overlap 256...



100%|█████████▉| 1048/1050 [01:37<00:00, 11.80it/s]
1052it [01:38, 11.69it/s]                          
1056it [01:38, 11.60it/s]
1060it [01:38, 11.52it/s]
1064it [01:39, 11.48it/s]
1068it [01:39, 11.50it/s]
1072it [01:39, 11.50it/s]
1076it [01:40, 11.56it/s]
1080it [01:40, 11.58it/s]
1084it [01:41, 11.56it/s]
1088it [01:41, 11.51it/s]
1092it [01:41, 11.36it/s]
1096it [01:42, 11.31it/s]
1100it [01:42, 11.36it/s]
1104it [01:42, 11.39it/s]
1108it [01:43, 11.38it/s]
1112it [01:43, 11.41it/s]
1116it [01:43, 11.58it/s]
1118it [01:43, 10.75it/s]


Inference completed in 104.63 seconds
Saved prediction to naip_test_prediction.tif


## Vectorize masks

In [14]:
output_path = "naip_test_prediction.geojson"
gdf = geoai.orthogonalize(masks_path, output_path, epsilon=2)

Processing 182 features...



Generated: 336, With features: 113:  32%|███▏      | 336/1050 [1:02:08<2:12:03, 11.10s/it]


Saving to naip_test_prediction.geojson...
Done!


## Visualize results

In [15]:
geoai.view_vector_interactive(output_path, tiles=test_raster_url)

In [19]:
geoai.create_split_map(
    left_layer=output_path,
    right_layer=test_raster_url,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_url,
)

TypeError: localtileserver.widgets.get_leaflet_tile_layer() got multiple values for keyword argument 'name'

In [17]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [18]:
from google.colab import output
output.disable_custom_widget_manager()

![image](https://github.com/user-attachments/assets/8dfcc69e-7a6c-408a-9fae-10b81b7d85dc)

In [20]:
# prompt: download folder output

import shutil
from google.colab import files

shutil.make_archive("output", 'zip', "output")
files.download("output.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>